# ¿Cómo desplegar la policía de Chicago para luchar eficazmente contra la delincuencia?

In [ ]:
import pandas as pd
import numpy  as np
import plotly.graph_objects as go
import branca
import geopandas
import folium

import plotly.express as px

from wordcloud import WordCloud #
from IPython.display import display



Han encontrado un conjunto de datos a disposición de la policía de Chicago de 2017 con información sobre los delitos cometidos en toda la ciudad.

Van a leer y ver su conjunto de datos. Este conjunto de datos se descarga de este sitio web. Contiene los incidentes delictivos denunciados (a excepción de los asesinatos, de los que existen datos para cada víctima) que se produjeron en la ciudad de Chicago en 2017.


In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/Alexander-Luna/Modulo2UIDE/main/Week2/Chicago_crime_data.csv',
                 dtype={'ID': object, 'beat_num': object})
df

Podemos ver arriba que la tabla contiene 22 columnas y hay 268.303 registros en total. A continuación se ofrece una breve descripción de cada columna:

|     Variable name    |                    Descripción de la variable                    |                                                                                                  Nota                                                                                                  |
|:--------------------:|:---------------------------------------------------------------:|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------:|
|          ID          |              Identificador único para el registro              |                                                                   Cada víctima en un solo caso de homicidio se asigna a un ID diferente                                                                  |
|      Case Number     | El número RD (División de Registros) del Departamento de Policía de Chicago |                                                  Único para el incidente. Múltiples IDs pueden compartir el mismo número de caso si el incidente es un caso de homicidio                                                 |
|         Date         |               Fecha en que ocurrió el incidente              |                                                                                Podría ser una mejor estimación para algunos registros                                                                               |
|         Block        |     La dirección parcialmente redactada donde ocurrió el incidente     |                                                                    La dirección redactada está en el mismo bloque que la dirección real                                                                    |
|         IUCR         |          El código de informe uniforme de crímenes de Illinois          |                                Vinculado directamente al tipo principal y la descripción del delito. Ver detalles [aquí](https://data.cityofchicago.org/widgets/c7ck-438e)                                |
|     Primary Type     |          La descripción principal del código IUCR          |      -                                                                                                                                                                                                  |
|      Description     |         La descripción secundaria del código IUCR         |     -                                                                                                                                                                                                   |
| Location Description |   Descripción de la ubicación donde ocurrió el incidente  |    -                                                                                                                                                                                                    |
|        Arrest        |                  Si se realizó un arresto                 |     -                                                                                                                                                                                                   |
|       Domestic       |          Si el incidente estaba relacionado con violencia doméstica         |                                                                La definición de relacionado con violencia doméstica se basa en la Ley de Violencia Doméstica de Illinois                                                               |
|       beat_num       |         El área de patrulla policial donde ocurrió el incidente        |          El área geográfica policial más pequeña - cada área de patrulla tiene un coche policial dedicado. Ver detalles [aquí](https://data.cityofchicago.org/Public-Safety/Boundaries-Police-Beats-current-/aerh-rz74)         |
|       District       |       El distrito policial donde ocurrió el incidente      | Tres a cinco áreas de patrulla conforman un sector policial y tres sectores conforman un distrito policial. Ver detalles [aquí](https://data.cityofchicago.org/Public-Safety/Boundaries-Police-Districts-current-/fthy-xz3r) |
|         Ward         |            La zona donde ocurrió el incidente            |                          Las zonas son distritos del consejo municipal. Ver detalles [aquí](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Wards-2015-/sp34-6z76)                          |
|    Community Area    |       El área comunitaria donde ocurrió el incidente       |                                     Ver detalles [aquí](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Community-Areas-current-/cauq-8yn6)                                    |
|       FBI Code       |   La clasificación del crimen según lo descrito en el NIBRS del FBI  |                         NIBRS significa Sistema Nacional de Informes de Incidentes Basados (NIBRS). Ver detalles [aquí](http://gis.chicagopolice.org/clearmap_crime_sums/crime_types.html)                         |
|       Latitude       |  La latitud de la ubicación donde ocurrió el incidente  |                                                   Esta ubicación se desplaza de la ubicación real para redacción parcial pero cae en el mismo bloque                                                  |
|       Longitude      |  La longitud de la ubicación donde ocurrió el incidente |       -

# Configuración de celda para las visualizaciones

In [ ]:
def enable_plotly_in_cell():
  import IPython
  from plotly.offline import init_notebook_mode
  display(IPython.core.display.HTML('''
        <script src="/static/components/requirejs/require.js"></script>
  '''))
  init_notebook_mode(connected=False) # modo offline

get_ipython().events.register('pre_run_cell', enable_plotly_in_cell)

## Fase 1: Investigando crímenes por tipo y descripción.

Para iniciar nuestro análisis vamos a explorar la relación entre las variables "primary_type" y "description". Como ambas variables son discretas podemos contar el número total de registros que pertenecen a categorías específicas. Tenga en cuenta que cada tipo de "primary_type" tiene su propio conjunto de descripciones que no se solapan. Es decir si dos delitos tienen diferentes "primary_type" entonces no tendrán la misma descripción por definición (a esto se le conoce como variables anidadas).

**Tarea 1**: Crea un gráfico que permita observar la cantidad de ocurrencias por delito.

**Pregunta 1**: Qué pueden identificar??

In [ ]:
#TAREA 1
# Contamos la frecuencia de repetición por delito
primary_type_counts = df['Primary Type'].value_counts().reset_index()
primary_type_counts.columns = ['Primary Type', 'Count']

#Gráfico BAR, con color por Primary Type, y template de colores ggplot2
fig = px.bar(primary_type_counts, x='Primary Type', y='Count', color='Primary Type',
             title="Cantidad de ocurrencias por tipo de delito en Chicago",
             width=1000, height=700,
             template="ggplot2"
             )
#Definición del layout, nombres de ejes x, y. Sin leyendas porque tenemos por color
fig.update_layout(xaxis_title='Tipo de Delito', yaxis_title='Número de Ocurrencias',
                  showlegend=False, hovermode='closest')

#Personalizar el texto de la leyenda Hover cuando se coloca el mouse en la barra
fig.update_traces(
    hovertemplate='Tipo de Delito: %{x}<br>Número de Ocurrencias: %{y}<extra></extra>'
)

# Configuraciones extra a la gráfica: ScrollZoom, botones para descargar en jpeg, tamaño y nombre del archivo
config = {
    "scrollZoom": True,
    "toImageButtonOptions": {
             'format': 'jpeg',
             'width':1400,
             'height':1400,
             'filename': "Número de Ocurrencias por delito"
             }}

fig.show(config =config)

>**RTA**:
>>* A partir del gráfico de barras, se puede identificar claramente que los tipos de delito más frecuentes en Chicago durante el periodo analizado son: THEFT (Robo), BATTERY (Agresión) y CRIMINAL DAMAGE (Daño Criminal).
>>* Estos tres tipos de delitos concentran una proporción significativamente alta de las ocurrencias delictivas.
>>* La elección del diagrama de barras fue adecuada ya que permite una comparación visual directa y eficiente de la frecuencia entre los distintos tipos de delitos, facilitando la identificación rápida de los delitos predominantes.

Como sabemos que estas dos variables son anidadas podemos desglosar aún más las frecuencias anteriores incluyendo la descripción.

**Tarea 2**: Escriba código utilizando la función groupby, para contar el número de casos en todas las combinaciones de ``Primary_type`` y ``Description``. A continuación, ordene los resultados en orden decreciente según el número de casos, y grafique en un gráfico de barras agrupado o apilados donde el eje x sea el Primary_type y las barras sean la Description.

**Pregunta 2**: ¿Cuáles son las descripciones más frecuentes de casos de robo (THEFT), lesiones (Battery) y daños penales (Criminal Damage) en Chicago?

In [ ]:
#Contar la cantidad de delitos por descripción
data_1 = df.groupby(["Primary Type", "Description"])["ID"].count().reset_index(name="count").sort_values(by="count", ascending=False).reset_index(drop=True)
data_1.head()


In [ ]:
#Valores únicos de descripción de tipo de delito
len(data_1["Description"].unique())

In [ ]:
# Coloquen el código aquí ####################
# Tarea 2
# Se grafican los top 20 delitos vs descripción, ya que en total se encontraton 310
fig = px.bar(data_1.head(20), x="Primary Type", y="count", color="Description", barmode="group",
             title="Cantidad de ocurrencias por tipo de delito AGRUPADOS en Chicago",
             width=1000, height=700,
             template="ggplot2",
             custom_data=["Description"]    #Para poder personalizar el hover
             )

#Definición del layout, nombres de ejes x, y. Sin leyendas porque tenemos por color
fig.update_layout(bargap=.3,
                  xaxis_title='Tipo de Delito', yaxis_title='Número de Ocurrencias',
                  showlegend=True, hovermode='closest')

#Personalizar el texto de la leyenda Hover cuando se coloca el mouse en la barra
fig.update_traces(width=0.4,
                  hovertemplate = 'Tipo de Delito: %{x}<br>Número de Ocurrencias: %{y}<br>Descripción: %{customdata[0]}<extra></extra>'
                  )

# Configuraciones extra a la gráfica: ScrollZoom, botones para descargar en jpeg, tamaño y nombre del archivo
config = {
    "scrollZoom": True,
    "toImageButtonOptions": {
             'format': 'jpeg',
             'width':1400,
             'height':1400,
             'filename': "Número de Ocurrencias por delito, agrupadas por Descripción"
             }}

fig.show(config =config)

**RTA**:
>* Del análisis de frecuencias, sabemos que hay 310 descripciones en total y presentarlas todas en el gráfico no es viable (se mostraron 20).
>* Descripciones más frecuentes:   Se encontraron las top 3 de la siguiente manera:

In [ ]:
# Descripciones más frecuentes THEFT
print("\nDescripciones más frecuentes para Tipo> THEFT")
data_filtrada = data_1[data_1["Primary Type"] == "THEFT"]
print(data_filtrada[["Description", "count"]].head(3))

# Descripciones más frecuentes BATTERY
print("\nDescripciones más frecuentes para Tipo> BATTERY")
data_filtrada = data_1[data_1["Primary Type"] == "BATTERY"]
print(data_filtrada[["Description", "count"]].head(3))

# Descripciones más frecuentes ASSAULT
print("\nDescripciones más frecuentes para Tipo> ASSAULT")
data_filtrada = data_1[data_1["Primary Type"] == "ASSAULT"]
print(data_filtrada[["Description", "count"]].head(3))

También podemos utilizar una herramienta de visualización conocida como nube de palabras (word cloud) para resumir las descripciones predominantes dentro de cada tipo primario. Una nube de palabras visualiza las palabras dentro de una colección de textos (en nuestro caso, los textos son todas las descripciones de un tipo primario específico) y el tamaño de cada palabra es proporcional a la frecuencia con que aparece en la base de datos. A continuación, construimos tres nubes de palabras para los 3 tipos de delitos primarios más frecuentes:

In [ ]:
# wordcloud para primary type definido por un rango o posición del crimen
def wordcloud_crime( df, rank, word):
    df_filter = df[df["Primary Type"]==df["Primary Type"].value_counts().index[rank]]
    text = ' '.join(df_filter['Description'])
    wordcloud = WordCloud(max_font_size=50, max_words=100, background_color="white").generate(text)
    # usamos en este caso matplotlib
    #plt.imshow(wordcloud)
    fig = px.imshow(wordcloud, binary_string=True)
    fig.update_layout(title=f'Nube de palabras para {word}')
    fig.show()

In [ ]:
word_i = df["Primary Type"].value_counts().index[0]
wordcloud_crime( df, 0, word_i)

In [ ]:
word_i = df["Primary Type"].value_counts().index[1]
wordcloud_crime( df, 1, word_i)

In [ ]:
word_i = df["Primary Type"].value_counts().index[2]
wordcloud_crime( df, 2, word_i)

**Pregunta 3**: ¿Cuáles son las palabras más comunes para describir estos tipos de casos?

**RTA:**

In [ ]:
import pandas as pd
from collections import Counter
import re

def analizar_palabras_frecuentes(df, tipos_crimen, top_n=5):
    resultados = {}

    # Lista de "stop words" comunes en inglés que no aportan significado
    stop_words = {'and', 'to', 'the', 'of', 'for', 'in', 'on', 'with', 'by', 'from'}

    for tipo in tipos_crimen:
        # Filtrar el dataframe por el tipo de crimen
        subset = df[df["Primary Type"] == tipo]

        word_counts = Counter()

        for _, row in subset.iterrows():
            # Limpiar la descripción: minúsculas y quitar caracteres no alfanuméricos
            desc = row['Description'].lower()
            palabras = re.findall(r'\w+', desc)

            # Contar palabras ignorando stop words y números aislados
            for p in palabras:
                if p not in stop_words and not p.isdigit():
                    # Multiplicamos por el 'count' para reflejar la frecuencia real
                    word_counts[p] += row['count']

        resultados[tipo] = word_counts.most_common(top_n)

    return resultados

# Ejecutar el análisis para los tipos solicitados
tipos = df["Primary Type"].value_counts().head(3).index.tolist()  # Obtener los 3 tipos más frecuentes
frecuencias = analizar_palabras_frecuentes(data_1, tipos)

# Mostrar resultados
for crime, words in frecuencias.items():
    print(f"\nTop palabras para {crime}:")
    for word, freq in words:
        print(f" - {word.upper()}: {freq} apariciones")


Como hemos visto con las nubes de palabras, parece que un determinado tipo de delito suele estar relacionado con ciertos tipos de lugares (por ejemplo, en casa, en tiendas).

**Tarea 3:** Creen una gráfica para investigar los patrones de delincuencia asociados a los tipos de lugares de los delitos (Location Description).

**Pregunta 4:** Basándote en los resultados, ¿qué tipos de lugares son más propensos a la delincuencia?

In [ ]:
# Tarea 3
# Contamos la frecuencia de repetición por descripción de ubicación
location_description_counts = df['Location Description'].value_counts().reset_index()
location_description_counts.columns = ['Location Description', 'Count']

# Tomar las 20 descripciones de ubicación más frecuentes para una mejor visualización
top_20_locations = location_description_counts.head(20)

# Gráfico BAR, con color por Location Description, y template de colores ggplot2
fig = px.bar(top_20_locations, x='Location Description', y='Count', color='Location Description',
             title="Cantidad de ocurrencias por tipo de ubicación de delito en Chicago (Top 20)",
             width=1200, height=700,
             template="ggplot2"
             )
# Definición del layout, nombres de ejes x, y. Sin leyendas porque tenemos por color
fig.update_layout(xaxis_title='Tipo de Ubicación', yaxis_title='Número de Ocurrencias',
                  showlegend=False, hovermode='closest')

# Personalizar el texto de la leyenda Hover cuando se coloca el mouse en la barra
fig.update_traces(
    hovertemplate='Tipo de Ubicación: %{x}<br>Número de Ocurrencias: %{y}<extra></extra>'
)

# Configuraciones extra a la gráfica: ScrollZoom, botones para descargar en jpeg, tamaño y nombre del archivo
config = {
    "scrollZoom": True,
    "toImageButtonOptions": {
             'format': 'jpeg',
             'width':1400,
             'height':1400,
             'filename': "Número de Ocurrencias por Ubicación de Delito"
             }}

fig.show(config =config)

**RTA**: Basándonos en los resultados, los tipos de lugares más propensos a la delincuencia son:

* STREET (Calle): Con el mayor número de ocurrencias, las calles son el lugar más frecuente para la actividad delictiva.
* RESIDENCE (Residencia) y APARTMENT (Apartamento): Estos dos lugares combinados representan una parte significativa de los delitos, indicando que los hogares también son sitios comunes de incidentes.
* SIDEWALK (Acera): Al igual que las calles, las aceras presentan un alto número de delitos.

Hasta ahora, hemos visto patrones delictivos vinculados con el "Tipo principal" y la "Descripción de la ubicación" por separado. Tiene sentido ver si una determinada combinación de tipo de delito y tipo de ubicación es frecuente o no. Sabemos que tanto el "Tipo de delito" como la "Descripción del lugar" son variables discretas. Por lo tanto, podemos utilizar una **tabla de contingencia** (tabla cruzada) para resumir el número total de incidentes que pertenecen a una combinación específica de valores de "Primary Type" y "Location Description".

A diferencia del análisis realizado con "Primary Type" y "Description", la `Location Description` y el `Primary Type` **NO** son variables anidadas. Por lo tanto, podemos utilizar la función `crosstab` de `pandas` para generar la tabla de contingencia de dos variables.

**Tarea 4**: La función `crosstab(var1, var2)` genera la tabla de contingencia para `var1` vs. `var2`. Utilice esta función para generar una tabla de contingencia para `Primary Type` vs `Location Description` en la que primero sólo se incluyan las 10 ubicaciones y tipos de delitos más frecuentes.

**Tarea 5**: Después use una figura de mapa de calor (Heatmap) a través del API graph_objects (go) de Plotly para visualizar la tabla de contingencia.

**Pregunta 5**: Basándose en la tabla de contingencia, ¿cuáles son los puntos calientes de los 10 tipos de delitos más frecuentes? ¿Son los mismos o no?

In [ ]:
# Tarea 4

# Obtener los 10 'Primary Type' más frecuentes
top_10_primary_types = df['Primary Type'].value_counts().head(10).index.tolist()

# Obtener las 10 'Location Description' más frecuentes
top_10_location_descriptions = df['Location Description'].value_counts().head(10).index.tolist()

# Filtrar el DataFrame para incluir solo estas 10 categorías principales de cada columna
df_filtered_crosstab = df[
    df['Primary Type'].isin(top_10_primary_types) &
    df['Location Description'].isin(top_10_location_descriptions)
]

# Generar la tabla de contingencia (crosstab)
contingency_table = pd.crosstab(
    df_filtered_crosstab['Primary Type'],
    df_filtered_crosstab['Location Description']
)

# Mostrar la tabla de contingencia
display(contingency_table)

In [ ]:
fig = go.Figure(data=go.Heatmap(
    z=contingency_table.values,
    x=contingency_table.columns,
    y=contingency_table.index,
    colorscale='Viridis',
    colorbar=dict(title='Número de Ocurrencias'),
    hovertemplate='<b>Tipo de Delito:</b> %{y}<br><b>Ubicación:</b> %{x}<br><b>Ocurrencias:</b> %{z}<extra></extra>'
))

fig.update_layout(
    title='Mapa de Calor de Delitos por Tipo y Ubicación (Top 10)',
    xaxis_title='Descripción de la Ubicación',
    yaxis_title='Tipo Principal de Delito',
    xaxis_nticks=len(contingency_table.columns) + 1,
    yaxis_nticks=len(contingency_table.index) + 1,
    width=1200,
    height=700
)

# Configuraciones extra a la gráfica: ScrollZoom, botones para descargar en jpeg, tamaño y nombre del archivo
config = {
    "scrollZoom": True,
    "toImageButtonOptions": {
             'format': 'jpeg',
             'width':1400,
             'height':1400,
             'filename': "Mapa de Calor de Delitos por Tipo y Ubicación"
             }}

fig.show(config=config)

>>**RTA**: Basándonos en el mapa de calor y la tabla de contingencia, podemos identificar los siguientes puntos calientes para los 10 tipos de delitos más frecuentes:

*   **STREET (Calle)**: Es un punto caliente significativo para varios delitos, incluyendo **THEFT (Robo)**, **BATTERY (Agresión)**, **CRIMINAL DAMAGE (Daño Criminal)**, **ROBBERY (Robo a Mano Armada)** y **MOTOR VEHICLE THEFT (Robo de Vehículo)**.
*   **RESIDENCE (Residencia)** y **APARTMENT (Apartamento)**: Son puntos calientes importantes para **BATTERY**, **THEFT**, **BURGLARY (Robo en Vivienda)**, **ASSAULT (Agresión)**, **DECEPTIVE PRACTICE (Prácticas Engañosas)** y **OTHER OFFENSE (Otros Delitos)**.
*   **PARKING LOT/GARAGE(NON.RESID.) (Estacionamiento/Garaje no residencial)**: Destaca como punto caliente para **THEFT** y **MOTOR VEHICLE THEFT**.
*   **SMALL RETAIL STORE (Pequeña Tienda Minorista)** y **RESTAURANT (Restaurante)**: Son puntos calientes para **THEFT**.

**¿Son los mismos o no?**

No, los puntos calientes no son completamente los mismos para todos los tipos de delitos. Si bien algunas ubicaciones como `STREET` y `RESIDENCE/APARTMENT` son consistentemente de alto riesgo para una variedad de crímenes, existen diferencias notables:

*   **STREET** es más general y aplica a delitos como robo, agresión y daño.
*   Las **RESIDENCE/APARTMENT** son más propensas a delitos relacionados con el hogar como hurto, agresión doméstica y allanamiento de morada.
*   Los **PARKING LOT/GARAGE** tienen una concentración más específica de robos y robos de vehículos.
*   Las **RETAIL STORES** y **RESTAURANTS** se asocian principalmente con el hurto.

Esto sugiere que las estrategias de despliegue policial deben ser diferenciadas, enfocándose en tipos específicos de delitos y sus ubicaciones más frecuentes, en lugar de un enfoque único para todos.

**Pregunta 6:** Con la información que ha descubierto hasta ahora como puede desplegar los policias de Chicago.



**RTA**: Basándonos en la información analizada hasta ahora, la estrategia para desplegar la policía de Chicago debería considerar los siguientes puntos clave:

1.  **Priorización de Delitos Más Frecuentes**: Enfocar recursos en la prevención y respuesta a los tipos de delitos más comunes: **THEFT (Robo)**, **BATTERY (Agresión)** y **CRIMINAL DAMAGE (Daño Criminal)**.

2.  **Despliegue Estratégico por Ubicación**:
    *   **Calles y Aceras (`STREET`, `SIDEWALK`)**: Mantener una presencia policial visible y patrullaje constante en estas áreas, ya que son puntos calientes generales para una amplia variedad de delitos, incluyendo robos, agresiones, daños criminales y robos de vehículos.
    *   **Residencias y Apartamentos (`RESIDENCE`, `APARTMENT`)**: Asignar recursos para responder eficazmente a incidentes relacionados con la violencia doméstica (`DOMESTIC BATTERY SIMPLE`), robos en viviendas (`BURGLARY`), agresiones y delitos de estafa, que son prevalentes en estos lugares.
    *   **Estacionamientos y Garajes (`PARKING LOT/GARAGE(NON.RESID.)`)**: Intensificar la vigilancia en estas áreas para prevenir `THEFT` (especialmente de vehículos) y `MOTOR VEHICLE THEFT`.
    *   **Tiendas Minoristas y Restaurantes (`SMALL RETAIL STORE`, `RESTAURANT`)**: Enfocar la atención en la prevención de `THEFT`, dado que estos establecimientos son objetivos frecuentes para este tipo de delito.

3.  **Focalización en Descripciones Específicas**:
    *   Para **THEFT**: Prestar especial atención a robos de bajo valor (`$500 AND UNDER`), de alto valor (`OVER $500`) y aquellos que ocurren `FROM BUILDING` (edificios) o en `RETAIL` (tiendas minoristas).
    *   Para **BATTERY**: Implementar programas de intervención y respuesta a la violencia `DOMESTIC`, y estar preparados para casos de agresión `SIMPLE` o `AGGRAVATED: HANDGUN`.
    *   Para **CRIMINAL DAMAGE**: Dirigir esfuerzos a la prevención de daños `TO PROPERTY` y `TO VEHICLE`.

4.  **Modelos de Patrullaje Diferenciados**: Evitar un enfoque único para todos los delitos. Diseñar rutas de patrullaje y asignaciones de personal que estén informadas por la naturaleza específica de los delitos y sus ubicaciones más probables. Por ejemplo, patrullas a pie en zonas comerciales para prevenir robos, y unidades de respuesta rápida para incidentes en residencias.

Esta estrategia inicial se centraría en la distribución espacial y la tipología delictiva, sentando las bases para una optimización futura con la inclusión de análisis temporales y otros factores.

## Fase 2: Investigando crímenes a través del tiempo.

Pasamos ahora a investigar la relación entre los incidentes delictivos y el tiempo; es decir, la variable `Date`. El tiempo es una de las dimensiones más importantes para construir un plan de despliegue eficaz. Dado que no podemos patrullar todos los lugares las 24 horas del día, los 7 días de la semana, debemos centrarnos en los periodos de tiempo con altos índices de delincuencia. La "fecha" nos proporciona una marca de tiempo para cada incidente, lo que nos permite contar cuántos incidentes se produjeron en un periodo de tiempo determinado. Como disponemos de los datos de un año, podemos empezar con el total mensual de incidentes para ver si ciertos meses son propensos a la delincuencia.

Vamos a cubrir diferentes temporalidades Día, Semana y Mes para tener varías perspectivas. Tenga en cuenta que en Chicago el clima juega un papel importante.

In [ ]:
# Primero convertir la columna Date a formato fecha
df["date_py"] = pd.to_datetime(df.Date, format="mixed")

**Tarea 6:** Gráfica el número de delitos en general a través del tiempo, agrupando por mes. Puedes usar pandas para agrupar por mes y contar el número de delitos.

**Pregunta 7:** ¿Qué meses tienen tasas de delincuencia relativamente más altas?

In [ ]:
df['month'] = df['date_py'].dt.month
monthly_crime_counts = df['month'].value_counts().sort_index().reset_index()
monthly_crime_counts.columns = ['Mes', 'Número de Delitos']

fig = px.bar(monthly_crime_counts, x='Mes', y='Número de Delitos',
             title='Número de Delitos por Mes en Chicago',
             labels={'Mes': 'Mes', 'Número de Delitos': 'Número de Delitos'},
             template='ggplot2')

fig.update_layout(xaxis=dict(tickmode='linear', dtick=1)) # Ensure all months are visible
fig.show()

**RTA**

**Tarea 7:** Gráfica el número de delitos en general a través del tiempo, agrupando por día. Puedes usar pandas para agrupar por día y contar el número de delitos.

**Pregunta 8:** ¿Sigue creyendo que febrero es la época en que la delincuencia es menos preocupante?

In [ ]:
df['day_of_month'] = df['date_py'].dt.day
daily_crime_counts = df['day_of_month'].value_counts().sort_index().reset_index()
daily_crime_counts.columns = ['Dia del Mes', 'Número de Delitos']

fig = px.bar(daily_crime_counts, x='Dia del Mes', y='Número de Delitos',
             title='Número de Delitos por Día del Mes en Chicago',
             labels={'Dia del Mes': 'Día del Mes', 'Número de Delitos': 'Número de Delitos'},
             template='ggplot2')

fig.update_layout(xaxis=dict(tickmode='linear', dtick=1)) # Ensure all days are visible
fig.show()

**RTA:**

**Tarea 8:** Gráfica el número de delitos en general a través del tiempo, agrupando por día de la semana. Puedes usar pandas para agrupar por día de la semana y contar el número de delitos.

**Pregunta 9:** ¿Cuáles son los días con mas y menos delitos?


In [ ]:
# Coloca el código aquí ############
# Tarea 8

**RTA**:

## Fase 3: Investigando crímenes por posición geográfica.

Otra dimensión importante que debemos considerar es la relación entre los incidentes delictivos y la ubicación geográfica. El aspecto técnico de este análisis puede verse como una profundización de lo que hemos aprendido previamente sobre visualizaciones de mapas.

Disponemos de las coordenadas geográficas aproximadas de cada incidente y, a partir de ellas, podemos explorar los patrones geográficos de la delincuencia en Chicago. Para identificar los focos geográficos de delincuencia, podemos dividir la ciudad de Chicago en regiones no superpuestas y contar el número total de casos en 2017 en cada región.En este caso, dividimos Chicago por distritos policiales y se cuentan los delitos por rondas policiales . A continuación, visualizamos los resultados en los mapas usando las librerías Plotly, Folium y geopandas:

**Pregunta 10:** Cuales puntos calientes identificaron?

Se usaron librerías adicionales ya que plotly presenta algunas complejidades y limitaciones para la visualización de mapas.

1. Con plotly se gráfica la distribución de puntos geo.

2. Se usa la librería geopandas para cargar los polígonos de las rondas policiales y cruzarlos con los puntos lon y lat y determinar asi el número de crimenes por polígono.

3. Seguidamente folium se utiliza para graficar esta información sobre un mapa de open street map, el cual es de acceso gratuito.

In [ ]:
# graficar una muestra de la distribución de puntos sobre el mapa
# de open street usando plotly para el tipo de delito mas comun
filter_df = df.loc[df['Primary Type'].isin(['THEFT', 'CRIMINAL DAMAGE'])]
#'BATTERY', 'ASSAULT', 'CRIMINAL DAMAGE']
fig = px.scatter_mapbox(filter_df, lat="Latitude",
                        lon="Longitude",
                        color='Primary Type',
                        zoom=8)

# Configurar el diseño del mapa
fig.update_layout(mapbox_style="open-street-map",
                  title='Distribución geo de delitos')

Para una funcionalidad más profunda involucrando un análisis espacial menos saturado y poder responder la pregunta 10, se usa las librerías geopandas y folium.

In [ ]:
#formatear la variable rondas policiales
df["beat_num"] = df["beat_num"].str.zfill(4)
# por rondas contar el numero de crimenes
beat_cn = df.groupby("beat_num")["ID"].count().reset_index(name="crime_count")
# Definir el esquema de color
min_cn, max_cn = beat_cn['crime_count'].quantile([0.01,0.99]).apply(round, 2)
# Crear la paleta de colores
colormap = branca.colormap.LinearColormap(
    colors=['white','yellow','orange','red','darkred'],
    #index=beat_cn['count'].quantile([0.2,0.4,0.6,0.8]),b
    vmin=min_cn,
    vmax=max_cn
)
colormap.caption="Total crimenes en Chicago por rondas policiales"

# cargar los poligonos de los sectores y limites en donde se hacen las rondas policiales
beat_orig = geopandas.read_file("https://raw.githubusercontent.com/Alexander-Luna/Modulo2UIDE/main/Week2/Boundaries_beat.geojson", driver = "GeoJSON")
# se interseca con los puntos
beat_data = beat_orig.join(beat_cn.set_index("beat_num"), how = "left", on = "beat_num")
beat_data.fillna(0, inplace = True)

In [ ]:
# Visualización interactiva del crimen por rondas policiales usando folium
m_crime = folium.Map(location=[41.88, -87.63],
                        zoom_start=12,
                        tiles="OpenStreetMap")
style_function = lambda x: {
    'fillColor': colormap(x['properties']['crime_count']),
    'color': 'black',
    'weight':2,
    'fillOpacity':0.5
}

stategeo = folium.GeoJson(
    beat_data.to_json(),
    name='Rondas policiales',
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['beat_num', 'crime_count'],
        aliases=['Ronda', 'Crimen total'],
        localize=True
    )
).add_to(m_crime)

colormap.add_to(m_crime)
m_crime

**RTA:**

**Pregunta 11**: Basándose en el análisis realizado hasta ahora, ¿cuál es su estrategia para el despliegue policial en función del tiempo, los lugares y los tipos de delitos?


**RTA:**